# Jev-style decisions on Cloudera AI — walkthrough

This notebook shows the library step by step: the input, what is sent to the model,
and what comes back. A decision takes three things — a piece of text (the *state*),
a *question*, and a fixed list of *options* — and returns a probability for each
option instead of written text. These field names follow the Jev interface, as
[SemIf-OpenJev](https://github.com/TheoLeeCJ/SemIf-OpenJev) does.

The model is a standard deployment on Cloudera AI Inference Service, called through
its OpenAI-compatible API. All claim notes here are synthetic development notes:
invented, with obviously fake names, policy numbers and vehicle numbers.

The next cell finds the project folder, so the notebook runs from `notebooks/` or
from the project root.

In [1]:
import copy
import json
import math
import os
import sys
from pathlib import Path

import pandas as pd

def find_project_root():
    here = Path.cwd()
    home = Path("/home/cdsw")
    candidates = [here, *here.parents, home]
    if home.is_dir():
        candidates += sorted(p for p in home.iterdir() if p.is_dir())
    for c in candidates:
        if (c / "jev" / "client.py").is_file() and (c / "data" / "questions.json").is_file():
            return c
    raise RuntimeError("Could not find the project folder (jev/client.py, data/questions.json).")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from jev.client import build_prompt_text, load_config, make_client, predict
from app.triage import calibrated, load_temperatures

with open("data/questions.json") as f:
    QUESTIONS = json.load(f)
with open("data/claims_dev.jsonl") as f:          # development notes only
    CLAIMS = {c["id"]: c for c in map(json.loads, f)}
print(f"{len(CLAIMS)} development notes loaded")

300 development notes loaded


## 1. Connect

The endpoint is configured only by environment variables, so the same notebook runs on
public or private cloud. This prints whether each one is set (never the address), then
makes one small test call.

In [2]:
for name in ("CAI_BASE_URL", "CAI_MODEL"):
    print(f"{name}: {'set' if os.environ.get(name) else 'NOT SET'}")

cfg = load_config()
client = make_client(cfg)
reply = client.chat.completions.create(
    model=cfg["model"], messages=[{"role": "user", "content": "Reply with OK."}], max_tokens=1)
print("Test call answered:", bool(reply.choices))

CAI_BASE_URL: set
CAI_MODEL: set


Test call answered: True


## 2. First decision

One development claim note, the team question and its four options, exactly as they
are in `data/questions.json`. `predict()` returns the chosen option, a probability for
each option, and the coverage: how much of the model's probability landed on the
option letters at all.

In [3]:
note = CLAIMS["claim-001"]["state"]
team = QUESTIONS["team"]
print("Note:", note, "\n")
print("Question:", team["question"], "\n")
for o in team["options"]:
    print(f"  {o['id']:<13} {o['description']}")

pred = predict(note, team["question"], team["options"], client, cfg)
print("\nChosen option:", pred.choice)
print("Coverage:", round(pred.coverage, 4))
pd.DataFrame({"probability": pred.probs}).round(4)

Note: FNOL. Caller Taylor Dummy, policy DEMO-315033, cb 555-0115. Rear-ended at low speed, bumper cracked on 2015 minivan, VIN DEMOVIN9625007310. Other driver at fault. Caller mentioned her neck is a bit sore and she might see a doctor. Photos uploaded. Est. loss $8,100. 

Question: Which team should handle this claim? Only if the note clearly fits more than one team, prefer fraud review, then coverage questions, then injury claims. 

  standard      Standard claims handling: routine vehicle or property damage, no injury, no red flags.
  injury        Injury claims: anyone hurt, medical treatment mentioned, or an attorney involved.
  fraud_review  Fraud review: details suggest the claim may not be genuine, such as a policy bought days before the loss, inconsistent accounts, or prior similar claims.
  coverage      Coverage questions: unclear whether the policy covers this loss, such as a lapsed policy, an excluded driver, or flood damage on a policy without flood cover.

Chosen option:

,probability
standard,0.0
injury,1.0
fraud_review,0.0
coverage,0.0


## 3. What's underneath

This is the exact prompt that was sent: the options are labelled A, B, C, D. The model
produces one token, and we read its scores for the option letters; no text is
generated. Below are the top 5 tokens from that one response, with their
log-probabilities (the model's scores on a log scale) and the matching probabilities.

In [4]:
print(build_prompt_text(note, team["question"], team["options"]))
top5 = sorted(pred.top_logprobs.items(), key=lambda kv: -kv[1])[:5]
pd.DataFrame([{"token": repr(t), "log-probability": lp, "probability": math.exp(lp)} for t, lp in top5])

Read the text and answer the question by choosing one option.

Text:
FNOL. Caller Taylor Dummy, policy DEMO-315033, cb 555-0115. Rear-ended at low speed, bumper cracked on 2015 minivan, VIN DEMOVIN9625007310. Other driver at fault. Caller mentioned her neck is a bit sore and she might see a doctor. Photos uploaded. Est. loss $8,100.

Question: Which team should handle this claim? Only if the note clearly fits more than one team, prefer fraud review, then coverage questions, then injury claims.

Options:
A. Standard claims handling: routine vehicle or property damage, no injury, no red flags.
B. Injury claims: anyone hurt, medical treatment mentioned, or an attorney involved.
C. Fraud review: details suggest the claim may not be genuine, such as a policy bought days before the loss, inconsistent accounts, or prior similar claims.
D. Coverage questions: unclear whether the policy covers this loss, such as a lapsed policy, an excluded driver, or flood damage on a policy without flood cove

,token,log-probability,probability
0,'B',-0.000042,9.999584e-01
1,'A',-10.250042,3.535602e-05
2,'C',-12.000042,6.143955e-06
3,'E',-18.125042,1.343985e-08
4,'D',-18.625042,8.151679e-09


## 4. Raw versus calibrated

Raw scores from one token sit near 100% for most notes, including many wrong
answers: instruction-tuned models put almost all their probability on their top next
token, so that score shows which answer the model prefers, not how often such answers
are right. Calibration rescales them with one temperature fitted on development
notes. The temperature is loaded exactly as the app loads it. The chosen option never
changes; only the confidence does.

In [5]:
temps = load_temperatures(QUESTIONS)
T = temps["team"]["temperature"]
print(f"Team temperature: {T:.3f} (from {temps['team']['run']}, fitted on {temps['team']['fit_n']} development notes)")
pd.DataFrame({"raw": pred.probs, "calibrated": calibrated(pred, T)}).round(4)

Team temperature: 4.781 (from results/dev/conditional_tiebreak, fitted on 150 development notes)


,raw,calibrated
standard,0.0,0.0961
injury,1.0,0.8205
fraud_review,0.0,0.0667
coverage,0.0,0.0167


## 5. Change a rule by editing a sentence

The options are plain sentences, so a rule can be changed by rewording one. Here we
copy the options and reword the *standard* option (in this notebook only;
`data/questions.json` is untouched), then rerun the same note.

This note and this rewording were chosen to show the effect: on this note the answer
switches from standard to injury. On other notes with a pain mentioned in passing,
the same rewording moved the probabilities less, and some did not change answer.

In [6]:
note5 = CLAIMS["claim-132"]["state"]
print("Note:", note5, "\n")

reworded = copy.deepcopy(team["options"])
print("Before:", reworded[0]["description"])
reworded[0]["description"] = ("Standard claims handling: routine vehicle or property damage where "
                              "nobody mentions any pain, soreness or a medical visit, and there are no red flags.")
print("After: ", reworded[0]["description"], "\n")

before = predict(note5, team["question"], team["options"], client, cfg)
after = predict(note5, team["question"], reworded, client, cfg)
print(f"Chosen before: {before.choice}   Chosen after: {after.choice}   "
      f"(correct answer in the demo data: {CLAIMS['claim-132']['team']})")
pd.DataFrame({"before": before.probs, "after": after.probs}).round(4)

Note: FNOL. Caller Kerry Testcase, policy DEMO-591262, cb 555-0157. Minor fender bender in a drive-through. 2021 pickup, VIN DEMOVIN6531353930 needs a new grille. Passenger went to urgent care later that day to get her back checked, just in case. Est. loss $3,400. 

Before: Standard claims handling: routine vehicle or property damage, no injury, no red flags.
After:  Standard claims handling: routine vehicle or property damage where nobody mentions any pain, soreness or a medical visit, and there are no red flags. 



Chosen before: standard   Chosen after: injury   (correct answer in the demo data: injury)


,before,after
standard,1.0,0.0012
injury,0.0,0.9988
fraud_review,0.0,0.0000
coverage,0.0,0.0000


## 6. A brand-new question

This question is not in the repository. Nothing was trained or configured for it: the
question and its two options are written here and sent with the note, the same way as
the team question. It runs on two development notes, one calm and one angry.

In [7]:
upset_q = "Does the customer sound upset?"
upset_options = [
    {"id": "yes", "description": "Yes, the customer sounds upset, angry or distressed."},
    {"id": "no", "description": "No, the customer sounds calm."},
]
rows = []
for cid, kind in (("claim-005", "calm"), ("claim-094", "angry")):
    p = predict(CLAIMS[cid]["state"], upset_q, upset_options, client, cfg)
    rows.append({"note": cid, "written to sound": kind, "answer": p.choice,
                 "P(yes)": round(p.probs["yes"], 4), "P(no)": round(p.probs["no"], 4),
                 "coverage": round(p.coverage, 4)})
    print(f"{cid} ({kind}): {CLAIMS[cid]['state'][:160]}...")
pd.DataFrame(rows)

claim-005 (calm): Insured Jordan Placeholder (DEMO-749137) called to report a loss. Scraped by another car in a parking lot while parked. Driver-side doors dented on 2015 minivan...
claim-094 (angry): Insured Drew Placeholder (DEMO-733228) called to report a loss. Insured furious about wait times, threatened to sue and post online. Loss is a cracked bathroom ...


,note,written to sound,answer,P(yes),P(no),coverage
0,claim-005,calm,no,0.0,1.0,1.0
1,claim-094,angry,yes,1.0,0.0,1.0


## 7. A small batch

The first 10 development notes through `predict()` for the team question, next to the
correct answers in the synthetic data. This is 10 synthetic notes, not an accuracy
measurement. The real evaluation (300 development notes and a separate 300-note test
set, with calibration and option-order checks) is in `eval/run_eval.py`, and its saved
results are in `results/`.

In [8]:
batch = []
for cid in list(CLAIMS)[:10]:
    c = CLAIMS[cid]
    p = predict(c["state"], team["question"], team["options"], client, cfg)
    batch.append({"note": cid, "predicted": p.choice, "correct": c["team"],
                  "match": p.choice == c["team"], "raw probability": round(p.probs[p.choice], 4)})
table = pd.DataFrame(batch)
print(f"{table['match'].sum()} of {len(table)} match the correct answer")
table

9 of 10 match the correct answer


,note,predicted,correct,match,raw probability
0,claim-001,injury,injury,True,1.0000
1,claim-002,injury,injury,True,1.0000
2,claim-003,coverage,coverage,True,1.0000
3,claim-004,injury,injury,True,1.0000
4,claim-005,standard,standard,True,1.0000
5,claim-006,standard,standard,True,0.9998
6,claim-007,fraud_review,standard,False,0.9949
7,claim-008,coverage,coverage,True,1.0000
8,claim-009,injury,injury,True,1.0000
9,claim-010,injury,injury,True,1.0000


## 8. Where to go next

- **The app:** `launch_app.py` starts the Workbench Application — paste a note, pick a
  question, and see the answer with calibrated confidence and a recommended action.
- **The evaluation:** `eval/run_eval.py` measures accuracy, balanced accuracy, option-order
  flips and calibration on the development or test notes; `eval/compare_runs.py`
  compares two saved runs.
- **The README:** setup on public and private cloud, results, and limitations.